# Phân tích thực nghiệm và trực quan hóa kết quả SMART Laptop

**Mục tiêu:** trực quan hóa kết quả thực nghiệm theo ba nhóm câu hỏi nghiên cứu được sử dụng trong bộ đánh giá:

- **RQ1 – Technical feasibility:** hệ thống có hiểu đúng yêu cầu, xử lý được ràng buộc và vận hành hết pipeline hay không?
- **RQ2 – Decision reliability:** sản phẩm được đề xuất có thuộc catalog và có đáp ứng các ràng buộc hay không?
- **RQ3 – Operational feasibility:** độ trễ, chi phí API, lượng token và số lần gọi API như thế nào?

Notebook này chỉ đọc kết quả benchmark đã chạy. **Không gọi Gemini, không chạy lại solver và không phát sinh chi phí API.**

> Chạy tuần tự từ trên xuống. Phần giải thích được đặt cạnh từng biểu đồ để có thể dùng notebook làm tài liệu ôn tập và trình bày báo cáo.

## 0. Phạm vi dữ liệu và quy tắc đọc kết quả

Bộ benchmark hiện có 300 query-turns:

- 300 kết quả Hybrid, gồm single-turn và multi-turn.
- 268 query-turns có cờ compare_with_baseline = 1, dùng để so sánh công bằng ba hệ thống.
- 32 query-turns multi-turn chỉ có Hybrid, không đưa vào phép so sánh ba hệ thống.
- Sau khi retry, hai baseline có 268/268 bản ghi hoàn thành.

**Quy tắc mẫu số:**

1. RQ1 đánh giá Hybrid trên toàn bộ 300 query-turns; riêng multi-turn dùng 32 trường hợp.
2. RQ2 và RQ3 so sánh ba hệ thống trên 268 query-turns chung.
3. Candidate Count Exact Rate chỉ tính trên truy vấn có expected_candidate_count được gán nhãn. Truy vấn thiếu nhãn không được coi là đúng.
4. OUT_OF_CATALOG chỉ có nghĩa là không ánh xạ được sản phẩm về catalog nghiên cứu; **không tự động đồng nghĩa với hallucination**.
5. Một số phép kiểm tra ràng buộc của LLM-only dựa trên thông số do chính LLM khai báo nếu sản phẩm không khớp catalog. Đây không phải bằng chứng kiểm chứng độc lập.

Notebook tính lại metric từ CSV đã đánh giá khi có thể, thay vì chỉ chép số từ summary.

## 1. Nạp thư viện và tìm dữ liệu

Mặc định notebook tìm:

- experiments/results/evaluated_300_merged.csv
- experiments/results/summary_300_merged.json
- experiments/test_queries_300_final.csv

Nếu tên hoặc vị trí file khác, sửa ba biến đường dẫn ở cell tiếp theo. Không dùng evaluated_300.csv cũ nếu mục tiêu là phân tích kết quả sau retry.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
})

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "experiments").is_dir():
            return candidate
    return Path.cwd()

ROOT = find_project_root()

# Thay bằng đường dẫn tuyệt đối nếu notebook không tự tìm thấy file.
EVALUATED_PATH_OVERRIDE = None
SUMMARY_PATH_OVERRIDE = None
BENCHMARK_PATH_OVERRIDE = None

def locate_file(filename, preferred_relative_paths, required=True):
    for rel in preferred_relative_paths:
        p = ROOT / rel
        if p.exists():
            return p
    for base in [ROOT / "experiments", ROOT]:
        if not base.exists():
            continue
        for p in base.rglob(filename):
            if any(part in {".git", "venv", ".venv", "__pycache__"} for part in p.parts):
                continue
            return p
    if required:
        raise FileNotFoundError(
            f"Không tìm thấy {filename}. Đặt file vào đúng vị trí trong experiments/ "
            "hoặc sửa biến *_PATH_OVERRIDE ở cell nạp dữ liệu."
        )
    return None

evaluated_path = Path(EVALUATED_PATH_OVERRIDE) if EVALUATED_PATH_OVERRIDE else locate_file(
    "evaluated_300_merged.csv", ["experiments/results/evaluated_300_merged.csv"], True
)
summary_path = Path(SUMMARY_PATH_OVERRIDE) if SUMMARY_PATH_OVERRIDE else locate_file(
    "summary_300_merged.json", ["experiments/results/summary_300_merged.json"], False
)
benchmark_path = Path(BENCHMARK_PATH_OVERRIDE) if BENCHMARK_PATH_OVERRIDE else locate_file(
    "test_queries_300_final.csv", ["experiments/test_queries_300_final.csv"], False
)

print("Project root :", ROOT)
print("Evaluated CSV:", evaluated_path)
print("Summary JSON :", summary_path if summary_path else "Không có; một số biểu đồ kiểm chứng ngoài sẽ được bỏ qua")
print("Benchmark CSV:", benchmark_path if benchmark_path else "Không có; dùng thông tin query trong evaluated CSV")

In [ ]:
df = pd.read_csv(evaluated_path, encoding="utf-8-sig")
summary = {}
if summary_path and summary_path.exists():
    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

benchmark = pd.read_csv(benchmark_path, encoding="utf-8-sig") if benchmark_path else None

def parse_bool(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n"}:
        return False
    return pd.NA

boolean_columns = [
    "compare_with_baseline", "expected_feasible", "tag_match_exact",
    "constraint_extraction_exact", "requirement_extraction_exact",
    "context_action_correct", "feasibility_detection_correct",
    "candidate_count_correct", "hard_constraint_satisfied",
    "product_valid", "independent_hard_constraint_satisfied",
    "independent_soft_constraint_violation",
]
for col in boolean_columns:
    if col in df.columns:
        df[col + "_b"] = df[col].map(parse_bool).astype("boolean")

numeric_columns = [
    "total_latency_ms", "estimated_api_cost_usd", "input_tokens",
    "output_tokens", "thinking_tokens", "total_tokens", "api_attempts",
    "nlu_api_attempts", "candidate_count", "expected_candidate_count",
    "catalog_explicit_hard_total_n", "catalog_explicit_soft_total_n",
]
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

SYSTEMS = ["llm_only", "catalog_grounded_llm", "hybrid_pipeline"]
SYSTEM_LABELS = {
    "llm_only": "LLM-only",
    "catalog_grounded_llm": "Catalog-grounded LLM",
    "hybrid_pipeline": "Hybrid",
}
SYSTEM_COLORS = {
    "llm_only": "#6C8EBF",
    "catalog_grounded_llm": "#E6A23C",
    "hybrid_pipeline": "#4A9D78",
}

comparison = df[df["compare_with_baseline_b"].fillna(False)].copy()
hybrid_all = df[df["system"].eq("hybrid_pipeline")].copy()
hybrid_multi = hybrid_all[hybrid_all["evaluation_scope"].astype(str).eq("multiturn")].copy()

print("Số bản ghi evaluated:", len(df))
display(df["system"].value_counts().rename_axis("system").to_frame("records"))
print("Số bản ghi so sánh chung:", len(comparison))
print("Số bản ghi Hybrid toàn bộ:", len(hybrid_all))
print("Số lượt multi-turn:", len(hybrid_multi))
print("Trạng thái sau retry:")
display(comparison.groupby(["system", "status"]).size().unstack(fill_value=0))

### 1.1. Cấu trúc benchmark

Biểu đồ mô tả phân bố nhóm truy vấn và nhãn khả thi. Đây là thông tin về **độ phủ của bộ thử nghiệm**, không phải chỉ số chất lượng mô hình.

- multi_turn được đánh giá riêng vì hai baseline không có đủ ngữ cảnh để so sánh trực tiếp.
- infeasible là các trường hợp kỳ vọng không tồn tại nghiệm thỏa mãn ràng buộc.
- Tỷ lệ feasible/infeasible ảnh hưởng đến cách diễn giải accuracy; cần nhìn số lượng từng lớp, không chỉ một phần trăm tổng hợp.

In [ ]:
qdf = benchmark.copy() if benchmark is not None else hybrid_all.drop_duplicates("query_id").copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
category_counts = qdf["category"].fillna("unknown").value_counts().sort_values()
axes[0].barh(category_counts.index, category_counts.values, color="#6C8EBF")
axes[0].set_title("Số query-turns theo nhóm")
axes[0].set_xlabel("Số query-turns")
for y, v in enumerate(category_counts.values):
    axes[0].text(v + 0.4, y, str(v), va="center")

feasible_col = qdf["expected_feasible"].map(parse_bool) if "expected_feasible" in qdf else qdf["expected_feasible_b"]
feasible_labels = feasible_col.map({True: "Feasible", False: "Infeasible"}).fillna("Missing")
feasible_counts = feasible_labels.value_counts().reindex(["Feasible", "Infeasible", "Missing"], fill_value=0)
axes[1].bar(feasible_counts.index, feasible_counts.values, color=["#4A9D78", "#D96C6C", "#999999"])
axes[1].set_title("Nhãn khả thi của benchmark")
axes[1].set_ylabel("Số query-turns")
for x, v in enumerate(feasible_counts.values):
    axes[1].text(x, v + 1, str(v), ha="center")
plt.tight_layout()
plt.show()
print("Tổng query-turns =", len(qdf))
print("Feasible =", int((feasible_labels == "Feasible").sum()),
      "| Infeasible =", int((feasible_labels == "Infeasible").sum()))

# 2. RQ1 – Technical feasibility

**Câu hỏi:** Kiến trúc Hybrid có thể tiếp nhận yêu cầu ngôn ngữ tự nhiên, chuyển chúng thành tags/ràng buộc có cấu trúc, xác định khả thi và xử lý hội thoại nhiều lượt hay không?

RQ1 đánh giá Hybrid trên toàn bộ 300 query-turns. Các metric không đo cùng một việc:

- **End-to-end completion:** pipeline có đi đến trạng thái kết quả hay không.
- **Exact match:** kết quả trích xuất có giống hoàn toàn nhãn kỳ vọng hay không.
- **Precision / Recall / F1:** mức khớp từng phần giữa tập phần tử dự đoán và tập kỳ vọng.
- **Feasibility accuracy:** dự đoán khả thi/vô nghiệm có đúng không.
- **Candidate count exact rate:** số lượng sản phẩm ứng viên có khớp nhãn hay không.
- **Multi-turn context action accuracy:** thao tác ADD/UPDATE/REPLACE có đúng ý định hội thoại hay không.

Không dùng một metric đơn lẻ để đại diện cho toàn bộ năng lực hệ thống.

## 2.1. End-to-end completion và trạng thái tối ưu

**End-to-end completion rate** ở đây là tỷ lệ query-turns mà pipeline trả về trạng thái kết quả, thay vì dừng ở ERROR.

Công thức: số query có trạng thái kết quả chia tổng số query.

- **OPTIMAL:** solver tìm được nghiệm tối ưu theo mô hình và mục tiêu đã thiết lập.
- **RELAXED:** hệ thống trả về phương án sau khi nới lỏng một phần điều kiện; không nên diễn giải như nghiệm thỏa mãn toàn bộ hard constraints ban đầu.

Vì vậy, biểu đồ tách trạng thái để tránh đánh đồng “pipeline hoàn thành” với “mọi ràng buộc được đáp ứng nghiêm ngặt”.

In [ ]:
status_counts = hybrid_all["status"].fillna("UNKNOWN").value_counts()
completed_statuses = {"OPTIMAL", "RELAXED"}
completion_n = int(hybrid_all["status"].isin(completed_statuses).sum())
completion_rate = completion_n / len(hybrid_all) if len(hybrid_all) else np.nan

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(["Completed pipeline", "Error / other"],
            [completion_n, len(hybrid_all)-completion_n],
            color=["#4A9D78", "#D96C6C"])
axes[0].set_ylabel("Query-turns")
axes[0].set_title(f"End-to-end completion = {completion_rate:.1%}")
for i, v in enumerate([completion_n, len(hybrid_all)-completion_n]):
    axes[0].text(i, v+2, str(v), ha="center")
axes[1].bar(status_counts.index.astype(str), status_counts.values,
            color=["#4A9D78" if x == "OPTIMAL" else "#E6A23C" if x == "RELAXED" else "#D96C6C" for x in status_counts.index])
axes[1].set_title("Phân bố trạng thái đầu ra Hybrid")
axes[1].set_ylabel("Query-turns")
for i, v in enumerate(status_counts.values):
    axes[1].text(i, v+2, str(v), ha="center")
plt.tight_layout()
plt.show()
print(f"Completion: {completion_n}/{len(hybrid_all)} = {completion_rate:.2%}")
print("Completion không đồng nghĩa với tỷ lệ thỏa mãn hard constraints.")

## 2.2. Exact match: Tag, Constraint và Full requirement

### Tag extraction exact rate
Tag là nhãn nhu cầu/nhóm sử dụng, ví dụ is_office_friendly hoặc is_gaming_friendly. Một query chỉ đúng khi **tập tag dự đoán trùng hoàn toàn tập tag kỳ vọng**. Thiếu hoặc sinh thừa một tag đều làm query đó sai.

### Constraint extraction exact rate
Constraint thường gồm bốn thành phần: field (thuộc tính), operator (toán tử), value (ngưỡng/giá trị) và type (hard/soft). Một query chỉ đúng nếu toàn bộ tập constraint trùng nhau sau chuẩn hóa giá trị. Ví dụ 15.000.000 và 15.000.000.0 được xem là cùng giá trị số; nhưng <= 20 triệu và <= 25 triệu là khác nhau.

### Full requirement exact rate
Đây là tiêu chí nghiêm ngặt hơn: tags, constraints **và preferences** đều phải trùng nhãn. Chỉ cần sai một thành phần thì cả query bị tính sai. Vì thế Full requirement exact thường thấp hơn Tag exact hoặc Constraint exact; đây không phải mâu thuẫn mà là hệ quả của điều kiện đồng thời.

In [ ]:
def bool_rate(series):
    values = series.dropna()
    return float(values.astype(bool).mean()) if len(values) else np.nan

rq1_exact = pd.Series({
    "Tag extraction exact": bool_rate(hybrid_all["tag_match_exact_b"]),
    "Constraint extraction exact": bool_rate(hybrid_all["constraint_extraction_exact_b"]),
    "Full requirement exact": bool_rate(hybrid_all["requirement_extraction_exact_b"]),
})
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(rq1_exact.index, rq1_exact.values*100, color=["#6C8EBF", "#E6A23C", "#4A9D78"])
ax.set_ylim(0, 105)
ax.set_ylabel("Tỷ lệ đúng (%)")
ax.set_title("RQ1 – Exact match của quá trình trích xuất yêu cầu")
ax.tick_params(axis="x", rotation=12)
for bar, value in zip(bars, rq1_exact.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f"{value:.1%}", ha="center")
plt.tight_layout()
plt.show()
display(rq1_exact.rename("rate").to_frame().assign(percent=lambda x: (x["rate"]*100).round(2)))

## 2.3. Precision, Recall và F1 – ý nghĩa và công thức

Exact match chỉ trả lời “toàn bộ tập có trùng không?”. Precision, Recall và F1 cho biết **sai ở đâu và sai theo hướng nào**.

Với một query:

- **TP (True Positive):** constraint được dự đoán và cũng có trong nhãn kỳ vọng.
- **FP (False Positive):** constraint được dự đoán nhưng không có trong nhãn kỳ vọng (sinh thừa điều kiện).
- **FN (False Negative):** constraint có trong nhãn kỳ vọng nhưng hệ thống bỏ sót.

Công thức:

- **Precision = TP / (TP + FP):** trong các constraint hệ thống đưa ra, tỷ lệ bao nhiêu là đúng. Precision thấp thường gợi ý hệ thống sinh thừa hoặc hiểu sai điều kiện.
- **Recall = TP / (TP + FN):** trong các constraint thực sự cần có, hệ thống tìm lại được bao nhiêu. Recall thấp thường gợi ý hệ thống bỏ sót yêu cầu.
- **F1 = 2 × Precision × Recall / (Precision + Recall):** trung bình điều hòa, cân bằng Precision và Recall.

**Cách tổng hợp trong evaluator:** tính Precision/Recall/F1 cho từng query rồi lấy trung bình trên 300 query (macro-average), không gộp tất cả constraint của mọi query thành một tập duy nhất (micro-average). Khi cả predicted và expected đều rỗng, evaluator quy ước P=R=F1=1; nếu chỉ một phía rỗng, metric bị phạt tương ứng.

Biểu đồ gồm cả tag và constraint. Đây là điểm trung bình theo query, không phải tỷ lệ query đúng hoàn toàn.

In [ ]:
prf_rows = []
for family, pcol, rcol, fcol in [
    ("Tag", "tag_precision", "tag_recall", "tag_f1"),
    ("Constraint", "constraint_precision", "constraint_recall", "constraint_f1"),
]:
    prf_rows.append({
        "Family": family,
        "Precision": pd.to_numeric(hybrid_all[pcol], errors="coerce").mean(),
        "Recall": pd.to_numeric(hybrid_all[rcol], errors="coerce").mean(),
        "F1": pd.to_numeric(hybrid_all[fcol], errors="coerce").mean(),
    })
prf = pd.DataFrame(prf_rows).set_index("Family")
ax = prf.plot(kind="bar", figsize=(10, 5), ylim=(0, 1.08), width=0.72)
ax.set_title("RQ1 – Precision, Recall và F1 trung bình theo query")
ax.set_ylabel("Điểm (0–1)")
ax.set_xlabel("")
ax.set_xticklabels(prf.index, rotation=0)
ax.legend(title="Metric", ncol=3, loc="lower right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=9)
plt.tight_layout()
plt.show()
display(prf.round(4))

## 2.4. Feasibility detection accuracy và confusion matrix

**Feasibility detection** là khả năng dự đoán một bộ yêu cầu có khả thi hay không.

- Nhãn thật (ground truth): expected_feasible.
- Dự đoán của hệ thống: hard_constraint_satisfied.
- Positive class trong confusion matrix dưới đây là **Feasible**.

Bốn ô:

- **TP:** thực tế feasible và hệ thống dự đoán feasible.
- **FN:** thực tế feasible nhưng hệ thống báo không feasible.
- **FP:** thực tế infeasible nhưng hệ thống dự đoán feasible. Đây là lỗi nguy hiểm vì có thể tư vấn sai.
- **TN:** thực tế infeasible và hệ thống dự đoán infeasible.

**Accuracy = (TP + TN) / tổng số trường hợp.**

Ngoài Accuracy, cần nhìn Sensitivity/Recall của lớp Feasible và Specificity/Recall của lớp Infeasible. Accuracy có thể gây hiểu lầm nếu hai lớp mất cân bằng; benchmark này có nhiều truy vấn feasible hơn infeasible.

In [ ]:
feas = hybrid_all[["expected_feasible_b", "hard_constraint_satisfied_b"]].dropna().copy()
y_true = feas["expected_feasible_b"].astype(bool)
y_pred = feas["hard_constraint_satisfied_b"].astype(bool)
labels = [True, False]  # Feasible trước, Infeasible sau
cm = np.array([[
    int(((y_true == labels[i]) & (y_pred == labels[j])).sum())
    for j in range(2)
] for i in range(2)])

tp, fn = cm[0, 0], cm[0, 1]
fp, tn = cm[1, 0], cm[1, 1]
accuracy = (tp+tn)/cm.sum()
sensitivity = tp/(tp+fn) if tp+fn else np.nan
specificity = tn/(tn+fp) if tn+fp else np.nan
precision_feasible = tp/(tp+fp) if tp+fp else np.nan

fig, ax = plt.subplots(figsize=(6.8, 5.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["Predicted Feasible", "Predicted Infeasible"])
ax.set_yticks([0, 1], ["Actual Feasible", "Actual Infeasible"])
ax.set_title("Confusion matrix – Feasibility detection")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black",
                fontsize=15, fontweight="bold")
fig.colorbar(im, ax=ax, label="Số query-turns")
plt.tight_layout()
plt.show()
print(f"Accuracy    = (TP+TN)/N = ({tp}+{tn})/{cm.sum()} = {accuracy:.2%}")
print(f"Sensitivity = TP/(TP+FN) = {tp}/{tp+fn} = {sensitivity:.2%}")
print(f"Specificity = TN/(TN+FP) = {tn}/{tn+fp} = {specificity:.2%}")
print(f"Precision   = TP/(TP+FP) = {tp}/{tp+fp} = {precision_feasible:.2%}")
print(f"TP={tp}, FN={fn}, FP={fp}, TN={tn}")

## 2.5. Candidate Count Exact Rate – mẫu số đúng

Metric này kiểm tra số lượng laptop ứng viên hệ thống báo có khớp expected_candidate_count hay không.

**Candidate Count Exact Rate = số query có candidate_count = expected_candidate_count / số query có expected_candidate_count.**

Không dùng toàn bộ 300 làm mẫu số, vì benchmark chỉ gán nhãn số lượng ứng viên cho một phần truy vấn. Ô thiếu nhãn không phải câu trả lời đúng; đó là trường hợp không đánh giá được.

Notebook tính lại trực tiếp từ hai cột số, không dùng candidate_count_correct nếu cột này đã đánh dấu trường hợp thiếu nhãn là đúng.

In [ ]:
expected_n = pd.to_numeric(hybrid_all["expected_candidate_count"], errors="coerce")
actual_n = pd.to_numeric(hybrid_all["candidate_count"], errors="coerce")
candidate_mask = expected_n.notna() & actual_n.notna()
candidate_rows = hybrid_all.loc[candidate_mask].copy()
candidate_rows["candidate_count_exact_recomputed"] = (
    actual_n.loc[candidate_mask].to_numpy() == expected_n.loc[candidate_mask].to_numpy()
)
candidate_correct_n = int(candidate_rows["candidate_count_exact_recomputed"].sum())
candidate_total_n = len(candidate_rows)
candidate_rate = candidate_correct_n/candidate_total_n if candidate_total_n else np.nan
candidate_wrong_n = candidate_total_n-candidate_correct_n

fig, ax = plt.subplots(figsize=(8, 4.5))
vals = [candidate_correct_n, candidate_wrong_n]
bars = ax.bar(["Khớp chính xác", "Không khớp"], vals, color=["#4A9D78", "#D96C6C"])
ax.set_title(f"Candidate Count Exact Rate = {candidate_rate:.2%} (n={candidate_total_n})")
ax.set_ylabel("Số query có nhãn")
ax.set_ylim(0, max(vals+[1])*1.2)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, str(v), ha="center")
plt.tight_layout()
plt.show()
print(f"Đúng: {candidate_correct_n}/{candidate_total_n}")
print(f"Sai : {candidate_wrong_n}/{candidate_total_n}")
print(f"Tỷ lệ: {candidate_rate:.2%}")
print("Truy vấn thiếu expected_candidate_count đã bị loại khỏi mẫu số.")

## 2.6. Multi-turn context action accuracy

Ở hội thoại nhiều lượt, hệ thống phải xác định người dùng đang:

- **ADD:** thêm điều kiện mới vào yêu cầu hiện tại.
- **UPDATE:** thay đổi một điều kiện đã có.
- **REPLACE:** thay thế yêu cầu cũ bằng yêu cầu mới.

**Context action accuracy = số lượt có action dự đoán trùng action kỳ vọng / số lượt multi-turn.**

Confusion matrix cho biết hệ thống nhầm ADD thành UPDATE hay nhầm REPLACE thành thao tác khác. Chỉ nhìn accuracy tổng thể sẽ không cho biết loại nhầm lẫn nào xảy ra. Mẫu số là 32 lượt multi-turn, không phải 300 lượt.

In [ ]:
mt = hybrid_multi.copy()
mt["expected_action"] = mt["expected_context_action"].fillna("UNKNOWN").astype(str)
mt["predicted_action"] = mt["context_action"].fillna("UNKNOWN").astype(str)
action_labels = sorted(set(mt["expected_action"]) | set(mt["predicted_action"]))
action_cm = pd.crosstab(mt["expected_action"], mt["predicted_action"]).reindex(
    index=action_labels, columns=action_labels, fill_value=0
)
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(action_cm.values, cmap="Purples")
ax.set_xticks(range(len(action_labels)), action_labels)
ax.set_yticks(range(len(action_labels)), action_labels)
ax.set_xlabel("Action dự đoán")
ax.set_ylabel("Action kỳ vọng")
ax.set_title("Confusion matrix – Multi-turn context action")
for i in range(len(action_labels)):
    for j in range(len(action_labels)):
        ax.text(j, i, str(action_cm.iloc[i, j]), ha="center", va="center",
                color="white" if action_cm.values[i, j] > action_cm.values.max()/2 else "black")
fig.colorbar(im, ax=ax, label="Số lượt hội thoại")
plt.tight_layout()
plt.show()
action_correct_n = int(mt["context_action_correct_b"].fillna(False).sum())
action_total_n = len(mt)
print(f"Context action accuracy = {action_correct_n}/{action_total_n} = {action_correct_n/action_total_n:.2%}")

## 2.7. Nguồn NLU và chất lượng theo nguồn

Hybrid có hai nguồn xử lý được ghi trong kết quả:

- **gemini_plus_regex:** Gemini kết hợp xử lý/chuẩn hóa bằng Regex.
- **regex_fallback:** dùng Regex dự phòng khi nhánh Gemini không hoàn tất theo cơ chế của hệ thống.

Biểu đồ thứ nhất mô tả mức độ hệ thống phụ thuộc vào từng nguồn. Biểu đồ thứ hai so sánh chất lượng trích xuất theo nguồn.

Nếu Full requirement exact thấp hơn ở Regex fallback, điều đó cho thấy fallback giúp pipeline tiếp tục chạy nhưng không đảm bảo chất lượng trích xuất tương đương. Đây là phân tích theo nhóm quan sát, không phải thí nghiệm ngẫu nhiên chứng minh Regex là nguyên nhân duy nhất gây giảm độ chính xác.

In [ ]:
source_counts = hybrid_all["nlu_source"].fillna("unknown").value_counts()
source_metrics = hybrid_all.groupby("nlu_source").agg(
    n=("query_id", "size"),
    tag_exact=("tag_match_exact_b", lambda s: s.dropna().astype(bool).mean()),
    constraint_exact=("constraint_extraction_exact_b", lambda s: s.dropna().astype(bool).mean()),
    full_exact=("requirement_extraction_exact_b", lambda s: s.dropna().astype(bool).mean()),
    constraint_f1=("constraint_f1", "mean"),
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].bar(source_counts.index.astype(str), source_counts.values, color=["#6C8EBF", "#E6A23C"])
axes[0].set_title("Số query theo nguồn NLU")
axes[0].set_ylabel("Số query")
for i, v in enumerate(source_counts.values):
    axes[0].text(i, v+2, str(v), ha="center")
source_metrics[["tag_exact", "constraint_exact", "full_exact"]].mul(100).plot(
    kind="bar", ax=axes[1], ylim=(0, 105)
)
axes[1].set_title("Exact rate theo nguồn NLU")
axes[1].set_ylabel("Tỷ lệ (%)")
axes[1].set_xlabel("")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=12)
axes[1].legend(["Tag exact", "Constraint exact", "Full requirement exact"])
plt.tight_layout()
plt.show()
display(source_metrics.round(4))

# 3. RQ2 – Decision reliability

**Câu hỏi:** kết quả tư vấn có đáng tin cậy không, đặc biệt về việc sản phẩm có thuộc catalog và có đáp ứng các ràng buộc của người dùng?

RQ2 chỉ dùng 268 query-turns có thể so sánh giữa cả ba hệ thống. Các biểu đồ tách riêng:

1. **Catalog validity:** có ánh xạ sản phẩm được đề xuất về sản phẩm trong catalog hay không.
2. **Hard-constraint satisfaction:** sản phẩm có đáp ứng yêu cầu bắt buộc hay không.
3. **Soft-constraint violation:** mức độ không đáp ứng ưu tiên mềm.

Ba khái niệm này không đồng nhất. Sản phẩm có thể nằm trong catalog nhưng vẫn vi phạm yêu cầu; sản phẩm ngoài catalog chưa chắc là sản phẩm không có thật.

## 3.1. Catalog validity và OUT_OF_CATALOG

**Catalog validity rate = số recommendation ánh xạ được về catalog / số recommendation hoàn thành.**

- **CATALOG_VALID:** evaluator tìm được sản phẩm tương ứng trong catalog theo quy tắc so khớp tên/model.
- **OUT_OF_CATALOG:** không ánh xạ được với catalog nghiên cứu.

Đây là phép đo độ bám catalog, không phải phép đo hallucination. Một model laptop có thật nhưng không có trong catalog vẫn có thể bị xếp OUT_OF_CATALOG.

Biểu đồ trả lời “hệ thống có giới hạn đầu ra vào catalog hay không?”, chứ không đủ để kết luận “LLM đã bịa sản phẩm”.

In [ ]:
catalog_rows = []
for system in SYSTEMS:
    g = comparison[comparison["system"].eq(system)].copy()
    completed = g[g["status"].eq("COMPLETED")] if system != "hybrid_pipeline" else g[g["status"].isin(["OPTIMAL", "RELAXED"])]
    valid_n = int(completed["recommendation_class"].eq("CATALOG_VALID").sum())
    out_n = int(completed["recommendation_class"].eq("OUT_OF_CATALOG").sum())
    denom = len(completed)
    catalog_rows.append({
        "system": SYSTEM_LABELS[system], "valid_n": valid_n, "out_n": out_n,
        "n": denom, "valid_rate": valid_n/denom if denom else np.nan,
        "out_rate": out_n/denom if denom else np.nan,
    })
catalog_table = pd.DataFrame(catalog_rows).set_index("system")
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(catalog_table))
valid_pct = catalog_table["valid_rate"].to_numpy()*100
out_pct = catalog_table["out_rate"].to_numpy()*100
ax.bar(x, valid_pct, label="Catalog valid", color="#4A9D78")
ax.bar(x, out_pct, bottom=valid_pct, label="Out of catalog", color="#D9A441")
ax.set_xticks(x, catalog_table.index)
ax.set_ylim(0, 110)
ax.set_ylabel("Tỷ lệ trên recommendation hoàn thành (%)")
ax.set_title("RQ2 – Catalog validity trên 268 query-turns chung")
ax.legend()
for i, row in enumerate(catalog_table.itertuples()):
    ax.text(i, 103, f"n={row.n}", ha="center", fontsize=9)
    ax.text(i, max(row.valid_rate*50, 2), f"{row.valid_rate:.1%}", ha="center",
            color="white" if row.valid_rate > .5 else "black")
plt.tight_layout()
plt.show()
display(catalog_table.round(4))

## 3.2. Đáp ứng toàn bộ hard requirements trên sản phẩm kiểm chứng được

Hard constraints là điều kiện bắt buộc, ví dụ ngân sách tối đa, cân nặng tối đa, GPU rời hoặc dung lượng pin tối thiểu.

Biểu đồ chỉ xét: (1) query có nhãn kỳ vọng feasible; (2) recommendation ánh xạ được về catalog; (3) sản phẩm được kiểm tra bằng thuộc tính trong catalog.

**All-hard satisfaction = số sản phẩm kiểm chứng được và thỏa toàn bộ hard requirements / số sản phẩm feasible ánh xạ được về catalog.**

**Cảnh báo cỡ mẫu:** LLM-only chỉ có rất ít sản phẩm ánh xạ được về catalog, nên tỷ lệ của nó có mẫu số nhỏ. Không so sánh trực tiếp tỷ lệ nếu bỏ qua số lượng mẫu. Trường hợp ngoài catalog bị loại khỏi phép kiểm chứng độc lập, không tự động tính là vi phạm.

In [ ]:
hard_rows = []
for system in SYSTEMS:
    g = comparison[comparison["system"].eq(system)].copy()
    feasible = g[g["expected_feasible_b"].fillna(False)]
    catalog_feasible = feasible[feasible["recommendation_class"].eq("CATALOG_VALID")]
    verified = catalog_feasible["independent_hard_constraint_satisfied_b"].dropna()
    hard_rows.append({
        "system": SYSTEM_LABELS[system],
        "satisfied_n": int(verified.astype(bool).sum()),
        "assessable_n": int(len(verified)),
        "rate": float(verified.astype(bool).mean()) if len(verified) else np.nan,
    })
hard_table = pd.DataFrame(hard_rows).set_index("system")
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(hard_table.index, hard_table["rate"]*100, color=[SYSTEM_COLORS[s] for s in SYSTEMS])
ax.set_ylim(0, 110)
ax.set_ylabel("Tỷ lệ đáp ứng toàn bộ hard requirements (%)")
ax.set_title("RQ2 – Hard-constraint satisfaction trên sản phẩm catalog-valid")
for bar, row in zip(bars, hard_table.itertuples()):
    if pd.notna(row.rate):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f"{row.rate:.1%}\n(n={row.assessable_n})", ha="center")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()
display(hard_table.round(4))

## 3.3. Explicit hard constraints: SATISFIED, VIOLATED và UNVERIFIABLE

- **SATISFIED:** có đủ thông tin để kiểm tra và ràng buộc được đáp ứng.
- **VIOLATED:** có đủ thông tin để kiểm tra và phát hiện vi phạm.
- **UNVERIFIABLE:** thiếu thông tin để kết luận; không được tự động coi là đúng hoặc sai.

Nguồn bằng chứng khác nhau:

- **LLM-only:** khi sản phẩm không khớp catalog, evaluator dựa trên thông số do LLM tự khai báo. Đây là claim-based, không phải xác minh độc lập.
- **Catalog-grounded LLM và Hybrid:** kiểm tra dựa trên catalog nghiên cứu.

Do đó, không nên kết luận LLM-only đạt độ chính xác hard constraint gần 99% theo nghĩa khách quan; con số đó chỉ phản ánh những trường hợp có thể đánh giá từ thông số mà LLM tự nêu.

In [ ]:
hard_status_rows = []
for system in SYSTEMS:
    g = comparison[comparison["system"].eq(system)].copy()
    feasible = g[g["expected_feasible_b"].fillna(False)]
    applicable = feasible[pd.to_numeric(feasible["catalog_explicit_hard_total_n"], errors="coerce").fillna(0).gt(0)]
    status_col = "claim_explicit_hard_constraint_status" if system == "llm_only" else "catalog_explicit_hard_constraint_status"
    statuses = applicable[status_col].fillna("UNVERIFIABLE").astype(str)
    counts = statuses.value_counts()
    hard_status_rows.append({
        "system": SYSTEM_LABELS[system],
        "SATISFIED": int(counts.get("SATISFIED", 0)),
        "VIOLATED": int(counts.get("VIOLATED", 0)),
        "UNVERIFIABLE": int(len(applicable)-counts.get("SATISFIED", 0)-counts.get("VIOLATED", 0)),
        "applicable_n": len(applicable),
        "evidence": "LLM self-claimed specs" if system == "llm_only" else "Independent catalog",
    })
hard_status = pd.DataFrame(hard_status_rows).set_index("system")
fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(hard_status))
for col, color in [("SATISFIED", "#4A9D78"), ("VIOLATED", "#D96C6C"), ("UNVERIFIABLE", "#B7B7B7")]:
    vals = hard_status[col].to_numpy()
    ax.bar(hard_status.index, vals, bottom=bottom, label=col, color=color)
    for i, v in enumerate(vals):
        if v > 0:
            ax.text(i, bottom[i]+v/2, str(v), ha="center", va="center", fontsize=9)
    bottom += vals
ax.set_ylabel("Số query feasible có hard constraint")
ax.set_title("RQ2 – Tình trạng kiểm chứng explicit hard constraints")
ax.legend(ncol=3)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()
display(hard_status)

## 3.4. Soft-constraint violation rate

Soft constraints là ưu tiên có thể đánh đổi, ví dụ thích máy nhẹ hơn, pin lâu hơn hoặc giá thấp hơn. Vi phạm soft constraint không nhất thiết làm recommendation vô hiệu.

**Soft violation rate = số trường hợp kiểm chứng được nhưng vi phạm ít nhất một soft constraint / số trường hợp soft constraint kiểm chứng được.**

Mẫu số không bao gồm UNVERIFIABLE. Biểu đồ đi kèm số mẫu n, vì tỷ lệ phần trăm không thể hiện mức độ bao phủ kiểm chứng.

Nguồn bằng chứng: LLM-only dựa trên thông số tự khai báo; hai hệ thống còn lại đối chiếu catalog.

In [ ]:
soft_rows = []
for system in SYSTEMS:
    g = comparison[comparison["system"].eq(system)].copy()
    soft_total = pd.to_numeric(g["catalog_explicit_soft_total_n"], errors="coerce").fillna(0)
    applicable = g[soft_total.gt(0)]
    status_col = "claim_explicit_soft_constraint_status" if system == "llm_only" else "catalog_explicit_soft_constraint_status"
    status = applicable[status_col].fillna("UNVERIFIABLE").astype(str)
    assessable = status.isin(["SATISFIED", "VIOLATED"])
    assessable_status = status[assessable]
    violated_n = int((assessable_status == "VIOLATED").sum())
    n = len(assessable_status)
    soft_rows.append({
        "system": SYSTEM_LABELS[system], "violated_n": violated_n,
        "assessable_n": n, "unverifiable_n": int((~assessable).sum()),
        "violation_rate": violated_n/n if n else np.nan,
    })
soft_table = pd.DataFrame(soft_rows).set_index("system")
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(soft_table.index, soft_table["violation_rate"]*100, color=[SYSTEM_COLORS[s] for s in SYSTEMS])
ax.set_ylabel("Tỷ lệ vi phạm soft constraints (%)")
ax.set_title("RQ2 – Soft-constraint violation rate trên trường hợp kiểm chứng được")
for bar, row in zip(bars, soft_table.itertuples()):
    if pd.notna(row.violation_rate):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{row.violation_rate:.1%}\n(n={row.assessable_n})", ha="center")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()
display(soft_table.round(4))

## 3.5. Kiểm chứng độc lập một mẫu sản phẩm LLM-only

Nếu summary có phần external_verification, notebook vẽ:

1. **Tình trạng xác minh model:** VERIFIED_REAL, EXACT_MODEL_NOT_CONFIRMED, NON_PRODUCT_RESPONSE.
2. **Tình trạng thông số:** MATERIAL_SPEC_ERROR, NO_MATERIAL_ERROR_FOUND, PARTIAL_CHECK_ONLY, UNVERIFIED_SPECS và các nhãn khác.

Phải phân biệt:

- EXACT_MODEL_NOT_CONFIRMED là chưa xác nhận được chính xác model, không tự động là hallucination.
- MATERIAL_SPEC_ERROR là lỗi thông số đáng kể trong nhóm sản phẩm đã xác minh có thật.
- Đây chỉ là mẫu kiểm chứng độc lập 52 đầu ra LLM-only, không đại diện tự động cho toàn bộ 268 đầu ra.

In [ ]:
external = summary.get("rq2_decision_reliability", {}).get("llm_only", {}).get("external_verification", {})
if external:
    verification_counts = external.get("verification_status_counts", {})
    spec_counts = external.get("spec_status_counts", {})
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    if verification_counts:
        names = list(verification_counts.keys())
        vals = [verification_counts[k] for k in names]
        axes[0].barh(names, vals, color="#6C8EBF")
        axes[0].set_title(f"External verification status (n={external.get('verified_output_n', sum(vals))})")
        axes[0].set_xlabel("Số đầu ra")
        for y, v in enumerate(vals):
            axes[0].text(v+0.2, y, str(v), va="center")
    if spec_counts:
        names2 = list(spec_counts.keys())
        vals2 = [spec_counts[k] for k in names2]
        axes[1].barh(names2, vals2, color=["#D96C6C" if k == "MATERIAL_SPEC_ERROR" else "#4A9D78" for k in names2])
        axes[1].set_title("External specification-check status")
        axes[1].set_xlabel("Số đầu ra")
        for y, v in enumerate(vals2):
            axes[1].text(v+0.2, y, str(v), va="center")
    plt.tight_layout()
    plt.show()
    print("Verified real products:", external.get("verified_real_product_n"))
    print("Material spec errors among verified real products:", external.get("material_spec_error_n"),
          "| rate =", external.get("material_spec_error_rate_on_verified_real_products"))
else:
    print("Không tìm thấy external_verification trong summary JSON; bỏ qua biểu đồ kiểm chứng ngoài.")

# 4. RQ3 – Operational feasibility

**Câu hỏi:** kiến trúc Hybrid có hiệu quả về mặt vận hành khi xét độ trễ, chi phí API, token và số lần gọi API hay không?

RQ3 chỉ sử dụng 268 query-turns chung. Các chỉ số:

- **Latency:** thời gian xử lý tổng cộng của một query-turn.
- **Mean:** nhạy với các trường hợp rất chậm.
- **Median:** thời gian của query ở vị trí giữa, ít bị outlier chi phối hơn.
- **P95:** 95% query có latency không vượt quá giá trị này; phản ánh phần đuôi chậm.
- **Estimated API cost:** chi phí API ước tính theo usage metadata và bảng giá đã cấu hình, không phải toàn bộ chi phí hạ tầng.
- **Tokens:** số token input/output/thinking.
- **API attempts:** baseline dùng api_attempts; Hybrid dùng nlu_api_attempts theo quy ước trong evaluator.

## 4.1. Latency: Mean, Median và P95

- **Mean latency:** tổng latency chia số query; dễ bị kéo lên bởi một số request rất chậm.
- **Median latency (P50):** một nửa query có latency thấp hơn hoặc bằng giá trị này; phản ánh trải nghiệm điển hình tốt hơn khi phân phối lệch.
- **P95 latency:** ngưỡng mà 95% query không vượt qua; hữu ích để đánh giá các trường hợp chậm ở phần đuôi.

Không kết luận một hệ thống nhanh hơn toàn diện chỉ từ Mean. Nếu Mean và Median tốt nhưng P95 kém hơn, hệ thống có thể nhanh trong đa số trường hợp nhưng vẫn có một nhóm request chậm.

In [ ]:
latency_rows = []
latency_distributions = {}
for system in SYSTEMS:
    g = comparison[comparison["system"].eq(system)]
    sec = pd.to_numeric(g["total_latency_ms"], errors="coerce").dropna()/1000
    latency_distributions[system] = sec
    latency_rows.append({
        "system": SYSTEM_LABELS[system], "n": len(sec),
        "Mean (s)": sec.mean(), "Median (s)": sec.median(),
        "P95 (s)": sec.quantile(0.95),
    })
latency_table = pd.DataFrame(latency_rows).set_index("system")
ax = latency_table[["Mean (s)", "Median (s)", "P95 (s)"]].plot(
    kind="bar", figsize=(11, 5), width=0.75
)
ax.set_title("RQ3 – So sánh latency trên 268 query-turns chung")
ax.set_ylabel("Thời gian (giây)")
ax.set_xlabel("")
ax.set_xticklabels(latency_table.index, rotation=10)
ax.legend(ncol=3)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=2, fontsize=8)
plt.tight_layout()
plt.show()
display(latency_table.round(3))

### Phân phối latency – outlier và độ phân tán

Boxplot mô tả phân phối thay vì chỉ một con số:

- Đường giữa hộp: median.
- Mép dưới/trên hộp: Q1 và Q3; 50% dữ liệu nằm trong hộp.
- Râu: phạm vi theo quy tắc boxplot.
- Điểm ngoài râu là outlier theo quy tắc này, không mặc nhiên là lỗi.

Trục tung dùng thang log để nhìn rõ cả request nhanh và chậm khi phân phối lệch.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
data = [latency_distributions[s].values for s in SYSTEMS]
bp = ax.boxplot(data, labels=[SYSTEM_LABELS[s] for s in SYSTEMS],
                showfliers=True, patch_artist=True)
for patch, system in zip(bp["boxes"], SYSTEMS):
    patch.set_facecolor(SYSTEM_COLORS[system])
    patch.set_alpha(0.65)
ax.set_yscale("log")
ax.set_ylabel("Latency (giây, log scale)")
ax.set_title("Phân phối latency theo hệ thống")
ax.grid(True, axis="y", which="both", alpha=0.25)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 4.2. Chi phí API: tổng chi phí và chi phí trên một recommendation

- **Total estimated API cost:** tổng chi phí ước tính để xử lý 268 query-turns; phù hợp so sánh chi phí của toàn bộ thí nghiệm có cùng số query.
- **Cost per completed recommendation:** tổng chi phí chia số recommendation hoàn thành; phù hợp ước tính chi phí cho một lần tư vấn.

Đơn vị là USD. Đây là **ước tính chi phí API** từ usage metadata và bảng giá được lưu trong code dự án. Không bao gồm máy chủ, database, lưu trữ, license solver, nhân công hay các chi phí vận hành khác.

In [ ]:
cost_rows = []
for system in SYSTEMS:
    g = comparison[comparison["system"].eq(system)]
    cost = pd.to_numeric(g["estimated_api_cost_usd"], errors="coerce").dropna()
    completed_n = int(g["status"].eq("COMPLETED").sum()) if system != "hybrid_pipeline" else int(g["status"].isin(["OPTIMAL", "RELAXED"]).sum())
    total_cost = cost.sum()
    cost_rows.append({
        "system": SYSTEM_LABELS[system], "n_cost_records": len(cost),
        "total_cost_usd": total_cost, "completed_n": completed_n,
        "cost_per_completed_usd": total_cost/completed_n if completed_n else np.nan,
    })
cost_table = pd.DataFrame(cost_rows).set_index("system")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(cost_table.index, cost_table["total_cost_usd"], color=[SYSTEM_COLORS[s] for s in SYSTEMS])
axes[0].set_title("Tổng estimated API cost")
axes[0].set_ylabel("USD")
axes[0].tick_params(axis="x", rotation=10)
for i, v in enumerate(cost_table["total_cost_usd"]):
    axes[0].text(i, v+max(cost_table["total_cost_usd"])*0.025, f"USD {v:.4f}", ha="center")
axes[1].bar(cost_table.index, cost_table["cost_per_completed_usd"], color=[SYSTEM_COLORS[s] for s in SYSTEMS])
axes[1].set_title("Estimated API cost / completed recommendation")
axes[1].set_ylabel("USD / recommendation")
axes[1].tick_params(axis="x", rotation=10)
for i, v in enumerate(cost_table["cost_per_completed_usd"]):
    axes[1].text(i, v+max(cost_table["cost_per_completed_usd"])*0.025, f"USD {v:.6f}", ha="center")
plt.tight_layout()
plt.show()
display(cost_table.round(8))

## 4.3. Token usage

- **Input tokens:** prompt và ngữ cảnh gửi đến mô hình.
- **Output tokens:** phần văn bản mô hình sinh ra.
- **Thinking tokens:** token dùng cho quá trình suy luận nội bộ nếu usage metadata có ghi nhận.
- **Total tokens:** tổng lượng token được ghi nhận.

Biểu đồ cột chồng thể hiện thành phần token trung bình trên mỗi query-turn. Token không đồng nghĩa trực tiếp với chi phí vì input, output và thinking có thể có đơn giá khác nhau. Cần đọc token usage cùng biểu đồ chi phí API.

In [ ]:
token_rows = []
for system in SYSTEMS:
    g = comparison[comparison["system"].eq(system)]
    token_rows.append({
        "system": SYSTEM_LABELS[system],
        "Input": pd.to_numeric(g["input_tokens"], errors="coerce").mean(),
        "Output": pd.to_numeric(g["output_tokens"], errors="coerce").mean(),
        "Thinking": pd.to_numeric(g["thinking_tokens"], errors="coerce").mean(),
        "Total": pd.to_numeric(g["total_tokens"], errors="coerce").mean(),
    })
token_table = pd.DataFrame(token_rows).set_index("system")
ax = token_table[["Input", "Output", "Thinking"]].plot(
    kind="bar", stacked=True, figsize=(10, 5),
    color=["#6C8EBF", "#4A9D78", "#E6A23C"]
)
ax.set_title("RQ3 – Token trung bình trên mỗi query-turn")
ax.set_ylabel("Tokens / query-turn")
ax.set_xlabel("")
ax.set_xticklabels(token_table.index, rotation=10)
ax.legend(title="Token type")
plt.tight_layout()
plt.show()
display(token_table.round(2))

## 4.4. Số lần gọi API trung bình

Số lần gọi API phản ánh mức độ lặp lại/fallback khi xử lý một query-turn.

- LLM-only và Catalog-grounded LLM: dùng trường api_attempts.
- Hybrid: dùng nlu_api_attempts, tức số lần gọi API ở thành phần NLU của Hybrid.

Không diễn giải đây là số lần gọi mọi dịch vụ trong toàn bộ ứng dụng; metric chỉ phản ánh số lần gọi API được logger ghi nhận theo quy ước trên.

In [ ]:
attempt_rows = []
for system in SYSTEMS:
    g = comparison[comparison["system"].eq(system)]
    col = "nlu_api_attempts" if system == "hybrid_pipeline" else "api_attempts"
    values = pd.to_numeric(g[col], errors="coerce").dropna()
    attempt_rows.append({
        "system": SYSTEM_LABELS[system], "n": len(values),
        "mean_attempts": values.mean(), "median_attempts": values.median(),
    })
attempt_table = pd.DataFrame(attempt_rows).set_index("system")
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(attempt_table.index, attempt_table["mean_attempts"],
              color=[SYSTEM_COLORS[s] for s in SYSTEMS])
ax.set_ylabel("Số lần gọi trung bình / query-turn")
ax.set_title("RQ3 – API attempts trung bình")
for bar, value in zip(bars, attempt_table["mean_attempts"]):
    ax.text(bar.get_x()+bar.get_width()/2, value+0.03, f"{value:.2f}", ha="center")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()
display(attempt_table.round(3))

# 5. Tổng hợp kết quả và giới hạn diễn giải

Bảng dưới đây tổng hợp các phép tính ở trên để hỗ trợ viết phần Results/Discussion. Khi đưa vào báo cáo, cần ghi rõ cỡ mẫu và nguồn bằng chứng.

In [ ]:
# Tóm tắt RQ1
rq1_summary = pd.DataFrame({
    "Metric": [
        "End-to-end completion",
        "Tag extraction exact",
        "Constraint extraction exact",
        "Full requirement exact",
        "Mean tag F1",
        "Mean constraint precision",
        "Mean constraint recall",
        "Mean constraint F1",
        "Feasibility detection accuracy",
        "Candidate count exact (labeled only)",
        "Multi-turn context action accuracy",
    ],
    "Value": [
        completion_rate,
        bool_rate(hybrid_all["tag_match_exact_b"]),
        bool_rate(hybrid_all["constraint_extraction_exact_b"]),
        bool_rate(hybrid_all["requirement_extraction_exact_b"]),
        pd.to_numeric(hybrid_all["tag_f1"], errors="coerce").mean(),
        pd.to_numeric(hybrid_all["constraint_precision"], errors="coerce").mean(),
        pd.to_numeric(hybrid_all["constraint_recall"], errors="coerce").mean(),
        pd.to_numeric(hybrid_all["constraint_f1"], errors="coerce").mean(),
        accuracy,
        candidate_rate,
        action_correct_n/action_total_n if action_total_n else np.nan,
    ],
})
rq1_summary["Value (%)"] = rq1_summary["Value"]*100
display(rq1_summary.round(4))
print("RQ1 denominator: 300 Hybrid query-turns; candidate count n =", candidate_total_n,
      "; multi-turn n =", action_total_n)

In [ ]:
# Tóm tắt RQ2 và RQ3
display(catalog_table.round(4))
print("Hard-constraint satisfaction trên catalog-valid feasible cases:")
display(hard_table.round(4))
print("Explicit hard-constraint evidence coverage:")
display(hard_status)
print("Soft-constraint violations:")
display(soft_table.round(4))
print("Latency:")
display(latency_table.round(3))
print("Cost:")
display(cost_table.round(8))
print("Token:")
display(token_table.round(2))
print("API attempts:")
display(attempt_table.round(3))

## 5.1. Những giới hạn cần ghi rõ trong báo cáo

1. **Candidate Count:** chỉ những truy vấn có expected candidate count mới được đưa vào mẫu số; không lấy 300 nếu thiếu nhãn.
2. **Catalog coverage:** OUT_OF_CATALOG không đồng nghĩa hallucination. Cần kiểm chứng độc lập để kết luận sản phẩm không tồn tại.
3. **Khác biệt bằng chứng:** LLM-only có một số phép kiểm tra dựa trên thông số tự khai báo; Catalog-grounded và Hybrid được đối chiếu catalog. Không trình bày chúng như cùng mức độ xác minh.
4. **Unverifiable:** thiếu thuộc tính catalog hoặc thiếu thông số không thể tính là thỏa mãn hay vi phạm. Báo cáo riêng số trường hợp không kiểm chứng được.
5. **External verification:** mẫu kiểm chứng ngoài chỉ gồm 52 đầu ra LLM-only, không đại diện tự động cho toàn bộ 268 đầu ra.
6. **Latency:** Mean, Median và P95 có thể cho thứ hạng khác nhau. Không chỉ chọn một con số thuận lợi.
7. **Cost:** là estimated API cost theo usage metadata; không phải tổng chi phí triển khai/kinh doanh.
8. **OPTIMAL/RELAXED:** pipeline hoàn thành không đồng nghĩa mọi yêu cầu gốc được đáp ứng cứng. Phân biệt completion, solver status và independent constraint verification.

### Gợi ý cách viết kết luận thận trọng

Trong phạm vi 300 query-turns của Hybrid và 268 query-turns so sánh chung, kết quả thực nghiệm cho thấy kiến trúc Hybrid có khả năng chuyển đổi yêu cầu tự nhiên thành biểu diễn cấu trúc với mức độ khớp cao, đồng thời cho thấy lợi thế về độ bám catalog, khả năng đáp ứng ràng buộc được kiểm chứng và chi phí API ước tính. Tuy nhiên, kết quả vẫn phụ thuộc vào chất lượng NLU, cơ chế cập nhật ngữ cảnh, độ bao phủ thuộc tính catalog và cách xây dựng nhãn đánh giá. Vì vậy, kết quả nên được diễn giải là bằng chứng thực nghiệm trong phạm vi benchmark, không phải bảo đảm tuyệt đối cho mọi yêu cầu laptop ngoài thực tế.

## 6. Xuất hình (tùy chọn)

Mặc định biểu đồ hiển thị trực tiếp trong notebook. Nếu muốn lưu hình để chèn Word/PowerPoint, thêm fig.savefig(...) ngay sau biểu đồ tương ứng.

Notebook không tự lưu hình để tránh ghi đè các hình đã có. Có thể tạo thư mục experiments/results/figures và lưu thủ công những biểu đồ cần đưa vào báo cáo.